In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

matches = pd.read_csv('matches.csv')
deliveries = pd.read_csv('deliveries.csv')
ipl_data = pd.read_csv('IPL (1).csv', engine='python', on_bad_lines='skip')

In [ ]:
matches.team1.head(10)

In [ ]:
# 1. Group by match and innings to get total runs per innings
# Note: Changed 'innings' to 'inning' to fix the KeyError
total_score_df = deliveries.groupby(['match_id', 'inning']).sum()['total_runs'].reset_index()

# 2. Filter for the 1st innings score
total_score_df = total_score_df[total_score_df['inning'] == 1]

# 3. Add 1 to get the 'Target' for the team batting second
total_score_df['target_score'] = total_score_df['total_runs'] + 1

# 4. Merge this back to your matches dataframe
final_df = matches.merge(total_score_df[['match_id', 'target_score']], left_on='id', right_on='match_id')

# Verify it worked
print(f"Success! Max target score is: {final_df['target_score'].max()}")
final_df.head()

In [ ]:
# Check unique teams to see current naming
print("Teams in matches:", matches['team1'].unique())

# Define the mapping for consistency
team_mapping = {
    'Kings XI Punjab': 'Punjab Kings',
    'Delhi Daredevils': 'Delhi Capitals',
    'Rising Pune Supergiants': 'Rising Pune Supergiant',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru',
    'Gujarat Lions': 'Gujarat Titans' # Optional: standardizing defunct/new teams if you prefer
}

# Apply the mapping to all relevant columns
columns_to_fix = ['team1', 'team2', 'toss_winner', 'winner']

for col in columns_to_fix:
    matches[col] = matches[col].replace(team_mapping)

# Do the same for deliveries dataset
deliveries['batting_team'] = deliveries['batting_team'].replace(team_mapping)
deliveries['bowling_team'] = deliveries['bowling_team'].replace(team_mapping)

print("\nTeam names standardized successfully!")

In [ ]:
# --- DYNAMICALLY FIXED ONE-MODEL OPTIMIZATION ---

# 1. Identify the correct dismissal column
# It's usually 'player_dismissed' or 'dismissal_kind'
possible_cols = ['player_dismissed', 'dismissal_kind', 'player_out']
wicket_col = next((col for col in possible_cols if col in ipl_data.columns), None)

if wicket_col:
    # We use 1 if there's a value, 0 if it's empty (NaN)
    ipl_data['is_wicket'] = ipl_data[wicket_col].apply(lambda x: 1 if pd.notnull(x) and x != "" else 0)
else:
    # Fallback if no column found (sets to 0 to prevent crash)
    ipl_data['is_wicket'] = 0
    print("Warning: Wicket column not found in ipl_data!")

# 2. Extract Stats from ipl_data (Innings 1)
# Note: Using 'total_runs' as it is more standard than 'team_runs'
runs_col = 'total_runs' if 'total_runs' in ipl_data.columns else 'team_runs'

innings_stats = ipl_data[ipl_data['innings'] == 1].groupby('match_id').agg({
    runs_col: 'sum', 
    'is_wicket': 'sum'
}).reset_index()

# 3. Rename for clarity
innings_stats.rename(columns={
    runs_col: 'target_score', 
    'is_wicket': 'wickets_lost'
}, inplace=True)

# 4. Merge with matches
final_df = matches.merge(innings_stats, left_on='id', right_on='match_id')

# 5. Calculate "Batting Strength Index" 
final_df['batting_index'] = final_df['target_score'] / (final_df['wickets_lost'] + 1)

# 6. Filter for final columns
final_features = ['city', 'team1', 'team2', 'toss_decision', 'target_score', 'wickets_lost', 'batting_index', 'winner']
final_df = final_df[final_features].dropna()

print("Features upgraded successfully! The model now handles wickets correctly.")
final_df.head()

In [ ]:
# Identify venues where city is missing
missing_city_venues = matches[matches['city'].isnull()]['venue'].unique()
print("Venues with missing cities:", missing_city_venues)

# Create a mapping for the missing ones (common in IPL datasets)
venue_to_city = {
    'M.Chinnaswamy Stadium': 'Bengaluru',
    'Dubai International Cricket Stadium': 'Dubai',
    'Sharjah Cricket Stadium': 'Sharjah',
    'Sheikh Zayed Stadium': 'Abu Dhabi'
}

# Fill the missing cities
matches['city'] = matches['city'].fillna(matches['venue'].map(venue_to_city))

# Check if any nulls remain
print("Remaining nulls in City:", matches['city'].isnull().sum())

In [ ]:
# Standardize City names
city_mapping = {
    'Bangalore': 'Bengaluru'
}
matches['city'] = matches['city'].replace(city_mapping)

# Double check the unique cities now
print(matches['city'].unique())

In [ ]:
# Select only columns known BEFORE or DURING the match
# We exclude 'player_of_match', 'result', 'result_margin', 'method', 'umpire1', 'umpire2'
required_columns = [
    'id',           # To merge with deliveries
    'city',         # Venue location
    'team1',        # Home/Away info
    'team2', 
    'toss_winner', 
    'toss_decision', 
    'venue', 
    'winner'        # This is our Target variable (what we want to predict)
]

# Create a filtered dataframe
match_df = matches[required_columns]

# Drop rows where the match was abandoned (No Winner)
match_df = match_df.dropna(subset=['winner'])

print("Data filtered. Remaining columns:", match_df.columns.tolist())

In [ ]:
# 1. Get 1st Inning Total from the 3rd Dataset (Ipl_data)
# We take the maximum 'team_runs' for the first inning of every match
first_inning_data = ipl_data[ipl_data['innings'] == 1].groupby('match_id')['team_runs'].max().reset_index()
first_inning_data.rename(columns={'team_runs': 'target_score'}, inplace=True)

# 2. Merge this with your cleaned 'matches' dataframe
final_df = matches.merge(first_inning_data, left_on='id', right_on='match_id')

# 3. Create 'Venue Average' feature (ground behavior)
venue_avg = final_df.groupby('venue')['target_score'].mean().reset_index()
venue_avg.rename(columns={'target_score': 'avg_venue_score'}, inplace=True)
final_df = final_df.merge(venue_avg, on='venue')

# 4. Create 'Toss Advantage' feature (1 if team1 won toss, 0 otherwise)
final_df['toss_winner_is_team1'] = (final_df['toss_winner'] == final_df['team1']).astype(int)

# 5. Filter for final columns needed for prediction
final_df = final_df[['city', 'team1', 'team2', 'toss_decision', 'toss_winner_is_team1', 'target_score', 'avg_venue_score', 'winner']]

# Drop any incomplete rows
final_df.dropna(inplace=True)

print("Feature Engineering Complete. Your model will now 'know' the ground's history and the first inning pressure.")
final_df.head()

In [ ]:
# --- FIX 1: RIVALRY & HOME ADVANTAGE ---

# 1. Define Home Cities for each team
home_cities = {
    'Mumbai Indians': 'Mumbai',
    'Kolkata Knight Riders': 'Kolkata',
    'Royal Challengers Bengaluru': 'Bengaluru',
    'Chennai Super Kings': 'Chennai',
    'Rajasthan Royals': 'Jaipur',
    'Delhi Capitals': 'Delhi',
    'Sunrisers Hyderabad': 'Hyderabad',
    'Punjab Kings': 'Chandigarh',
    'Gujarat Titans': 'Ahmedabad',
    'Lucknow Super Giants': 'Lucknow'
}

# 2. Feature: Is Team 1 playing at home?
final_df['team1_home_ad'] = final_df.apply(lambda x: 1 if home_cities.get(x['team1']) == x['city'] else 0, axis=1)

# 3. Feature: Head-to-Head (H2H) Win Ratio
# This calculates how often Team 1 beats Team 2 historically
def get_h2h(row):
    teams = sorted([row['team1'], row['team2']])
    temp_df = final_df[(final_df['team1'].isin(teams)) & (final_df['team2'].isin(teams))]
    team1_wins = len(temp_df[temp_df['winner'] == row['team1']])
    total_matches = len(temp_df)
    return team1_wins / total_matches if total_matches > 0 else 0.5

final_df['h2h_ratio'] = final_df.apply(get_h2h, axis=1)

print("Fix 1 Complete: Rivalry and Home Advantage features added.")

In [ ]:
# --- ADVANCED FEATURE ENGINEERING: TEAM FORM ---

# Calculate Win Ratio for Team 1 and Team 2
# We calculate total matches played and total matches won by each team
all_teams = pd.concat([final_df['team1'], final_df['team2']]).unique()
team_stats = []

for team in all_teams:
    played = len(final_df[(final_df['team1'] == team) | (final_df['team2'] == team)])
    won = len(final_df[final_df['winner'] == team])
    win_ratio = won / played if played > 0 else 0
    team_stats.append({'team': team, 'win_ratio': win_ratio})

team_stats_df = pd.DataFrame(team_stats)

# Map these ratios back to our final_df
final_df = final_df.merge(team_stats_df, left_on='team1', right_on='team', how='left').rename(columns={'win_ratio': 'team1_win_ratio'})
final_df = final_df.merge(team_stats_df, left_on='team2', right_on='team', how='left').rename(columns={'win_ratio': 'team2_win_ratio'})

# Drop extra columns from the merge
final_df.drop(['team_x', 'team_y'], axis=1, inplace=True)

print("Optimization 1 Complete: Team Form (Win Ratio) added as a feature.")
final_df.head()

In [ ]:
# --- STEP: CALCULATE WICKETS LOST ---
# 1. Identify the wicket column in your deliveries data (ipl_data)
possible_cols = ['player_dismissed', 'dismissal_kind', 'player_out']
wicket_col = next((col for col in possible_cols if col in ipl_data.columns), None)

if wicket_col:
    # Count wickets: 1 if a player was dismissed, 0 otherwise
    ipl_data['is_wicket'] = ipl_data[wicket_col].apply(lambda x: 1 if pd.notnull(x) and x != "" else 0)
    
    # Sum wickets per match for Innings 1
    wicket_stats = ipl_data[ipl_data['innings'] == 1].groupby('match_id')['is_wicket'].sum().reset_index()
    wicket_stats.rename(columns={'is_wicket': 'wickets_lost'}, inplace=True)
    
    # 2. Merge this into your final_df
    # Ensure 'id' matches the match_id from your matches/final_df
    if 'wickets_lost' not in final_df.columns:
        final_df = final_df.merge(wicket_stats, left_index=True, right_on='match_id', how='left')
        # Fill any missing values with 0 (in case of a match with 0 wickets)
        final_df['wickets_lost'] = final_df['wickets_lost'].fillna(0)
        print("Success: 'wickets_lost' column added.")
else:
    print("Error: Could not find a wicket/dismissal column in ipl_data.")

In [ ]:
# --- FIX 2: UPDATED ENCODING & SCALING ---
from sklearn.preprocessing import StandardScaler

# 1. Select the BEST columns (Include our new Fix 1 features)
columns_to_keep = [
    'city', 'team1', 'team2', 'toss_decision', 'toss_winner_is_team1', 
    'target_score', 'avg_venue_score', 'team1_win_ratio', 'team2_win_ratio', 
    'team1_home_ad', 'h2h_ratio', 'winner', 'wickets_lost'
]
final_df = final_df[columns_to_keep]

# 2. One-Hot Encode the remaining text
final_df_encoded = pd.get_dummies(final_df, columns=['city', 'team1', 'team2', 'toss_decision'])

# 3. Define X and y
X = final_df_encoded.drop('winner', axis=1)
y = (final_df['winner'] == final_df['team1']).astype(int)

# 4. SCALE the numerical values so the 'target_score' (200) doesn't 
# overwhelm the 'win_ratio' (0.6)
scaler = StandardScaler()
numerical_cols = ['target_score', 'avg_venue_score', 'team1_win_ratio', 'team2_win_ratio', 'h2h_ratio']
X[numerical_cols] = scaler.fit_transform(X[numerical_cols])

print("Fix 2 Complete: Features scaled and Rivalry data integrated.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.histplot(final_df['target_score'], bins=20, kde=True, color='skyblue')
plt.title('Distribution of 1st Inning Scores')
plt.xlabel('Target Score')
plt.ylabel('Frequency')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x='toss_decision', data=final_df, palette='magma')
plt.title('Toss Decision: Bat vs Field')
plt.show()

In [ ]:
# --- THE FINAL MASTER PREPROCESSING (Use this ONLY) ---
from sklearn.preprocessing import StandardScaler

# 1. THE REASON FOR THE DROP: Ensure we keep the 'wickets_lost' feature!
columns_to_keep = [
    'city', 'team1', 'team2', 'toss_decision', 'toss_winner_is_team1', 
    'target_score', 'avg_venue_score', 'team1_win_ratio', 'team2_win_ratio', 
    'team1_home_ad', 'h2h_ratio', 'wickets_lost', 'winner'
]

# Use ONLY the columns we need
final_df = final_df[columns_to_keep]

# 2. ENCODE (Turn names into numbers)
final_df_encoded = pd.get_dummies(final_df, columns=['city', 'team1', 'team2', 'toss_decision'])

# 3. DEFINE X AND Y
X = final_df_encoded.drop('winner', axis=1)
y = (final_df['winner'] == final_df['team1']).astype(int)

# 4. SCALE (The "Equalizer")
scaler = StandardScaler()
# We scale all numbers to be between -3 and +3 so the AI doesn't get confused
num_cols = ['target_score', 'avg_venue_score', 'team1_win_ratio', 'team2_win_ratio', 'h2h_ratio', 'wickets_lost']
X[num_cols] = scaler.fit_transform(X[num_cols])

print("Master Preprocessing Complete. 'Wickets' and 'Win Ratios' are locked in and scaled.")

In [ ]:
# --- STEP C: XGBOOST TRAINING ---
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# These specific parameters reached your peak accuracy
param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1],
    'max_depth': [3, 5, 7]
}

grid_search = GridSearchCV(XGBClassifier(eval_metric='logloss'), 
                           param_grid, cv=3, scoring='accuracy')

print("Training the optimized model...")
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print(f"\nREACHED ACCURACY: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("Best Settings Found:", grid_search.best_params_)

In [ ]:
import pickle
import os

# 1. Save all 4 assets
pickle.dump(best_model, open('ipl_model.pkl', 'wb'))
pickle.dump(scaler, open('scaler.pkl', 'wb'))
pickle.dump(list(X.columns), open('columns.pkl', 'wb'))

# Create the historical summary for the UI
h2h_summary = final_df.groupby(['team1', 'team2', 'winner']).size().reset_index(name='count')
pickle.dump(h2h_summary, open('h2h_stats.pkl', 'wb'))

print("--- EXPORT SUCCESSFUL ---")
print(f"Files saved in: {os.getcwd()}")
print("Confirmed files: ipl_model.pkl, scaler.pkl, columns.pkl, h2h_stats.pkl")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 1. Get feature importance from the best model
importance = best_model.feature_importances_
feature_names = X.columns

# 2. Create a DataFrame for visualization
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importance})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False).head(10)

# 3. Plot
plt.figure(figsize=(10, 6))
plt.barh(feature_importance_df['Feature'], feature_importance_df['Importance'], color='teal')
plt.xlabel('Importance Score')
plt.title('Top 10 Factors Predicting an IPL Win')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# --- THE FINAL CORRECTED PREDICTOR ENGINE ---

def get_live_prediction():
    print("--- IPL Match Winner Predictor ---")
    
    # 1. Collect inputs from the user
    city = input("Enter City: ")
    team1 = input("Enter Team 1 (Batting First): ")
    team2 = input("Enter Team 2 (Chasing): ")
    toss_decision = input("Enter Toss Decision (bat/field): ")
    target_score = int(input("Enter Target Score: "))
    wickets_lost = int(input("Enter Wickets Lost: "))

    # 2. Prepare the input for the AI
    input_data = pd.DataFrame(columns=X.columns)
    input_data.loc[0] = 0
    
    # One-Hot Encoding
    try:
        input_data[f'city_{city}'] = 1
        input_data[f'team1_{team1}'] = 1
        input_data[f'team2_{team2}'] = 1
        input_data[f'toss_decision_{toss_decision}'] = 1
    except KeyError:
        print("\nError: Spelling mistake in Team or City names!")
        return

    # 3. ADD THE MISSING "BRAIN" FEATURES (H2H & Home Advantage)
    # Calculate H2H Ratio for these two specific teams
    teams_pair = sorted([team1, team2])
    h2h_matches = final_df[(final_df['team1'].isin(teams_pair)) & (final_df['team2'].isin(teams_pair))]
    team1_wins = len(h2h_matches[h2h_matches['winner'] == team1])
    h2h_val = team1_wins / len(h2h_matches) if len(h2h_matches) > 0 else 0.5
    input_data['h2h_ratio'] = h2h_val

    # Calculate Home Advantage
    home_cities = {'Mumbai Indians': 'Mumbai', 'Chennai Super Kings': 'Chennai', 'Kolkata Knight Riders': 'Kolkata', 
                   'Royal Challengers Bengaluru': 'Bengaluru', 'Rajasthan Royals': 'Jaipur', 'Delhi Capitals': 'Delhi', 
                   'Sunrisers Hyderabad': 'Hyderabad', 'Punjab Kings': 'Chandigarh', 'Gujarat Titans': 'Ahmedabad', 'Lucknow Super Giants': 'Lucknow'}
    input_data['team1_home_ad'] = 1 if home_cities.get(team1) == city else 0

    # Assume Toss Winner (If decision is 'bat', Team 1 won. If 'field', Team 2 won and chose to bowl)
    input_data['toss_winner_is_team1'] = 1 if toss_decision == 'bat' else 0

    # 4. Set the numerical values
    input_data['target_score'] = target_score
    input_data['wickets_lost'] = wickets_lost
    input_data['avg_venue_score'] = final_df[final_df['city'] == city]['avg_venue_score'].mean()
    input_data['team1_win_ratio'] = final_df[final_df['team1'] == team1]['team1_win_ratio'].iloc[0]
    input_data['team2_win_ratio'] = final_df[final_df['team2'] == team2]['team2_win_ratio'].iloc[0]

    # 5. SCALE the inputs (Now including h2h_ratio!)
    num_cols = ['target_score', 'avg_venue_score', 'team1_win_ratio', 'team2_win_ratio', 'h2h_ratio', 'wickets_lost']
    input_data[num_cols] = scaler.transform(input_data[num_cols])

    # 6. GET THE PROBABILITY
    probabilities = best_model.predict_proba(input_data)[0]
    
    print("\n" + "="*35)
    print(f"PREDICTION: {team1} vs {team2}")
    print(f"{team1} Win Probability: {probabilities[1]*100:.2f}%")
    print(f"{team2} Win Probability: {probabilities[0]*100:.2f}%")
    print("="*35)

get_live_prediction()

In [ ]:
!pip freeze > requirements.txt